In [1]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from config import (
    LATENT_DIM, WGAN_EPOCHS, WGAN_BATCH_SIZE, N_CRITIC, GP_LAMBDA,
    WGAN_LR, WGAN_BETA1, WGAN_BETA2,
    N_SYNTHETIC_PER_CLASS, N_CLASSES, CLASS_NAMES,
    SYNTHETIC_DIR, MODELS_DIR, METRICS_DIR, FIGURES_DIR,
    RANDOM_SEED, ELECTRODE_NAMES, TMIN, TMAX, N_TIME_SUBSAMPLE, N_CWT_SCALES,
)
from preprocessing import preprocess_subject, split_by_class
from models.wgan_gp import build_generator, build_critic, WGANGP
from evaluate import plot_wgan_losses

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

SUBJECT = 1
print(f'TensorFlow {tf.__version__}  |  GPUs: {len(tf.config.list_physical_devices("GPU"))}')

TensorFlow 2.13.1  |  GPUs: 0


In [2]:
gen    = build_generator(LATENT_DIM)
critic = build_critic()

print('=' * 80)
print('GENERATOR')
print('=' * 80)
gen.summary(line_length=80)

print('\n' + '=' * 80)
print('CRITIC')

print('=' * 80)
critic.summary(line_length=80)

GENERATOR
Model: "generator"
________________________________________________________________________________
 Layer (type)                       Output Shape                    Param #     
 z (InputLayer)                     [(None, 100)]                   0           
                                                                                
 dense (Dense)                      (None, 8192)                    819200      
                                                                                
 batch_normalization (BatchNormali  (None, 8192)                    32768       
 zation)                                                                        
                                                                                
 re_lu (ReLU)                       (None, 8192)                    0           
                                                                                
 reshape (Reshape)                  (None, 4, 4, 512)               0           

In [3]:
X, y = preprocess_subject(SUBJECT, session='T')
class_data = split_by_class(X, y)

print(f'Dataset shape : {X.shape}')
for c, arr in class_data.items():
    print(f'  Class {c} ({CLASS_NAMES[c]:12s}): {len(arr)} trials')

  Loading subject 1 session T … 

c:\Users\vchagant\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


288 trials  shape=(50, 375, 5)
Dataset shape : (288, 50, 375, 5)
  Class 0 (Left Hand   ): 72 trials
  Class 1 (Right Hand  ): 72 trials
  Class 2 (Both Feet   ): 72 trials
  Class 3 (Tongue      ): 72 trials


In [ ]:
import time

DEMO_EPOCHS = 10
DEMO_BATCH  = 32   # smaller than WGAN_BATCH_SIZE=100 so all 72 samples/class fit

# NOTE: saves with _demo suffix — never overwrites real 300-epoch trained samples
DEMO_SUFFIX = '_demo'

all_synthetic  = {}
all_log_rows   = []

for cls_idx in range(N_CLASSES):
    X_cls = class_data[cls_idx]
    print(f'\n──── Class {cls_idx}: {CLASS_NAMES[cls_idx]} ({len(X_cls)} samples) ────')

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(X_cls.astype(np.float32))
        .shuffle(1000, seed=RANDOM_SEED)
        .batch(DEMO_BATCH, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )

    gen_c    = build_generator(LATENT_DIM)
    critic_c = build_critic()
    wgan_c   = WGANGP(gen_c, critic_c)
    wgan_c.compile(
        g_optimizer=tf.keras.optimizers.legacy.Adam(WGAN_LR, WGAN_BETA1, WGAN_BETA2),
        c_optimizer=tf.keras.optimizers.legacy.Adam(WGAN_LR, WGAN_BETA1, WGAN_BETA2),
    )

    for epoch in range(1, DEMO_EPOCHS + 1):
        c_vals, g_vals = [], []
        for batch in dataset:
            m = wgan_c.train_step(batch)
            c_vals.append(float(m['critic_loss']))
            g_vals.append(float(m['generator_loss']))
        mean_c, mean_g = np.mean(c_vals), np.mean(g_vals)
        if epoch % max(1, DEMO_EPOCHS // 5) == 0 or epoch == 1:
            print(f'  Epoch {epoch:3d}/{DEMO_EPOCHS} │ '
                  f'C: {mean_c:+8.4f}  G: {mean_g:+8.4f}')
        all_log_rows.append({
            'class_idx': cls_idx, 'class_name': CLASS_NAMES[cls_idx],
            'epoch': epoch, 'critic_loss': mean_c, 'generator_loss': mean_g,
        })

    z         = tf.random.normal([N_SYNTHETIC_PER_CLASS, LATENT_DIM])
    synthetic = gen_c(z, training=False).numpy()
    all_synthetic[cls_idx] = synthetic

    np.save(str(SYNTHETIC_DIR / f'synthetic_s{SUBJECT:02d}_c{cls_idx}{DEMO_SUFFIX}.npy'), synthetic)
    print(f'  {N_SYNTHETIC_PER_CLASS} synthetic samples saved (demo only)')

print('\nDemo training complete.')


In [ ]:
log_df = pd.DataFrame(all_log_rows)
save_path = str(FIGURES_DIR / f'wgan_losses_s{SUBJECT:02d}.png')
plot_wgan_losses(log_df, SUBJECT, save_path=save_path)

log_df.to_csv(str(METRICS_DIR / f'wgan_training_log_s{SUBJECT:02d}.csv'), index=False)
print('Log saved.')

In [ ]:
time_axis = np.linspace(TMIN, TMAX, N_TIME_SUBSAMPLE)

for cls_idx in range(N_CLASSES):
    real_samples = class_data[cls_idx][:5]     
    fake_samples = all_synthetic[cls_idx][:5]  

    fig, axes = plt.subplots(2, 5, figsize=(18, 5))
    for row, (label, samples) in enumerate(
        [('Real', real_samples), ('Generated', fake_samples)]
    ):
        for col, s in enumerate(samples):
            ax = axes[row, col]
            ax.pcolormesh(
                time_axis, np.arange(1, N_CWT_SCALES + 1),
                s[:, :, 1],  
                cmap='RdYlBu_r', vmin=-1, vmax=1, shading='auto'
            )
            ax.invert_yaxis()
            ax.set_xlabel('Time (s)' if row == 1 else '')
            if col == 0: ax.set_ylabel(label)
            ax.set_title(f'EEG Channel 2 (C3)' if (row == 0 and col == 2) else '')

    plt.suptitle(f'{CLASS_NAMES[cls_idx]} — Real (top) vs Generated (bottom)',
                 fontsize=13)
    out = str(FIGURES_DIR / f'real_vs_gen_s{SUBJECT:02d}_c{cls_idx}.png')
    plt.tight_layout()
    plt.savefig(out, dpi=150)
    plt.show()
    plt.close()
    print(f'  Saved → {os.path.basename(out)}')

In [ ]:
syn_X = np.concatenate([all_synthetic[c] for c in range(N_CLASSES)], axis=0)
syn_y = np.concatenate(
    [np.full(N_SYNTHETIC_PER_CLASS, c) for c in range(N_CLASSES)]
).astype(np.int32)

# Save with _demo suffix — does NOT overwrite real trained synthetic_combined_s01.npz
combined = SYNTHETIC_DIR / f'synthetic_combined_s{SUBJECT:02d}{DEMO_SUFFIX}.npz'
np.savez(str(combined), X=syn_X, y=syn_y)

print(f'Combined shape : {syn_X.shape}  labels: {np.unique(syn_y, return_counts=True)}')
print(f'Saved → {combined.name}')
print(f'\nReal trained samples are safe in synthetic_combined_s{SUBJECT:02d}.npz')
